# Feature Aggregation

This notebook merges all extracted feature files into a single dataset for machine learning.

**Input:**
- Individual feature pickle files from `features_mlcrowd/`

**Output:**
- `features_master.pkl`: Consolidated dataframe with all features


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Set display options
pd.set_option('display.max_columns', None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FEATURES_DIR = DATA_DIR / "features_mlcrowd"
OUTPUT_FILE = FEATURES_DIR / "features_master.pkl"

print(f"Features Directory: {FEATURES_DIR}")
print(f"Output File: {OUTPUT_FILE}")

# List available feature files
if FEATURES_DIR.exists():
    # Text embeddings (features_08) live in ../text_embeddings_mlcrowd/ and are attached by
    # build_text_master.ipynb; excluded here by name as well in case a copy ever lands in this folder.
    feature_files = sorted([f for f in FEATURES_DIR.glob("*.pkl")
                            if "master" not in f.name and "embedding" not in f.name])
    print(f"\nFound {len(feature_files)} feature files:")
    for f in feature_files:
        print(f" - {f.name}")
else:
    print(f"Directory not found: {FEATURES_DIR}")

In [ ]:
# Load and merge all feature files
if not feature_files:
    raise FileNotFoundError("No feature files found to merge.")

# Start with the first file
print(f"Loading base file: {feature_files[0].name}")
df_master = pd.read_pickle(feature_files[0])
print(f"  Shape: {df_master.shape}")

# Merge remaining files
for file_path in feature_files[1:]:
    print(f"\nMerging: {file_path.name}")
    df_next = pd.read_pickle(file_path)
    print(f"  Shape: {df_next.shape}")
    
    # Check for duplicate columns (excluding keys)
    common_cols = [c for c in df_next.columns if c in df_master.columns and c not in ['symbol', 'date']]
    if common_cols:
        print(f"  Warning: Duplicate columns found (keeping original): {common_cols}")
        df_next = df_next.drop(columns=common_cols)
    
    # Merge on symbol and date
    # Using outer merge to keep all data
    df_master = pd.merge(df_master, df_next, on=['symbol', 'date'], how='outer')

print(f"\n{'='*40}")
print(f"Merge Complete")
print(f"{'='*40}")
print(f"Final Shape: {df_master.shape}")
print(f"Unique Symbols: {df_master['symbol'].nunique()}")
print(f"Date Range: {df_master['date'].min()} to {df_master['date'].max()}")

In [ ]:
# Drop a few variables to avoid multicollinearity
cols_to_drop = ['n_bullish', 'n_bearish', 'bullish_ratio', 'bearish_ratio', 'total_labeled',
                'after_hours_volume', 'market_hours_volume', 'raw_volume']

for c in cols_to_drop:
    if c in df_master.columns:
        df_master = df_master.drop(columns=[c])
        print(f"Dropped column: {c}")

In [ ]:
# Inspect the merged dataframe
print("Columns in master dataframe:")
print(list(df_master.columns))

# Check for nulls
print("\nMissing values per column:")
print(df_master.isnull().sum()[df_master.isnull().sum() > 0])

# Save to pickle
print(f"\nSaving master feature file to: {OUTPUT_FILE}")
df_master.to_pickle(OUTPUT_FILE)
print("✓ Saved successfully!")
